**Лабораторная работа №3**

Создать сеть на базе LSTM используя TensorFlow (Keras). Сеть должна принимать на вход текстовый файл и на его базе генерировать свою абракадабру. Отчет должен содержать кроме кода, обучающий файл и результат генерации.


In [1]:
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
with open('base_text.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

# Разбиение текста на токены (слова, знаки препинания)
tokens = re.findall(r'\b\w+\b|[.,!?;:]', text)
# Словарь
vocab = sorted(set(tokens))
word_to_idx = {w:i for i,w in enumerate(vocab)}
idx_to_word = {i:w for i,w in enumerate(vocab)}
vocab_size = len(vocab)
print("Количество токенов:", len(tokens), "\nКоличество уникальных токенов:", vocab_size)
# Количество слов в контексте
seq_length = 10
step = 1
sequences, next_words = [], []
for i in range(0, len(tokens)-seq_length, step):
    seq = tokens[i:i+seq_length]
    nxt = tokens[i+seq_length]
    sequences.append([word_to_idx[w] for w in seq])
    next_words.append(word_to_idx[nxt])

X = np.array(sequences)
y = to_categorical(next_words, vocab_size)
print("X shape:", X.shape, "\ny shape:", y.shape)

print("Число обучающих примеров:", len(sequences))

Количество токенов: 3758 
Количество уникальных токенов: 1536
X shape: (3748, 10) 
y shape: (3748, 1536)
Число обучающих примеров: 3748


In [7]:
# Построение модели LSTM
embedding_dim = 128
# Индекс слова -> вектор
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=seq_length),
    LSTM(256, dropout=0.2, recurrent_dropout=0.2),
    Dense(vocab_size, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')

# Обучение модели
early_stop = EarlyStopping(monitor='loss', patience=5, min_delta=0.001)
history = model.fit(X, y, batch_size=64, epochs=100, callbacks=[early_stop], verbose=1)

Epoch 1/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - loss: 6.7856
Epoch 2/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 6.2223
Epoch 3/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - loss: 6.1052
Epoch 4/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - loss: 5.9884
Epoch 5/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - loss: 5.8606
Epoch 6/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - loss: 5.7364
Epoch 7/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 5.6252
Epoch 8/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 5.5331
Epoch 9/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - loss: 5.4312
Epoch 10/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - loss: 5.2969
Epoch 11/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - loss: 5.1682
Epoch 12/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 5.0048
Epoch 13/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 4.8528
Epoch 14/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - loss: 4.6840
Epoch 15/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - lo

In [8]:
# Генерация текста из слов и знаков
# Чем выше температура, тем более «случайно» выбираются слова
def generate_text(start_string, gen_length=100, temperature=0.8):
    # Разбиение начальной строки на токены
    start_tokens = re.findall(r'\b\w+\b|[.,!?;:]', start_string.lower())
    generated = start_tokens[:]
    context = start_tokens[-seq_length:]
    for _ in range(gen_length):
        # Преобразование токенов в индексы
        indices = [word_to_idx.get(w, 0) for w in context]
        if len(indices) < seq_length:
            indices = [0]*(seq_length - len(indices)) + indices
        else:
            indices = indices[-seq_length:]
        x_pred = np.array([indices])
        preds = model.predict(x_pred, verbose=0)[0]
        # Корректировка распределения температурой
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))
        next_idx = np.random.choice(vocab_size, p=preds)
        next_word = idx_to_word[next_idx]
        generated.append(next_word)
        context.append(next_word)
        context = context[-seq_length:]
    return ' '.join(generated)

# «Затравка» (основа для генерации текста) — случайный фрагмент из обучающего файла
start_idx = np.random.randint(0, len(tokens)-seq_length)
seed = ' '.join(tokens[start_idx:start_idx+seq_length])
print("Затравка:", seed)
result = generate_text(seed, gen_length=100, temperature=2.0)
print("\nСГЕНЕРИРОВАННЫЙ ТЕКСТ:")
print(result)

Затравка: : витька ведь деда , конечно , любил дед был

СГЕНЕРИРОВАННЫЙ ТЕКСТ:
: витька ведь деда , конечно , любил дед был дед что миг собранье . запинался молчал , свою несносно голубятник , им ему всех другие домоуправу первый прилетим , слава у клад ничего , ведь четверки домоуправу б глазел убежит отвечал ванька : витьку цветами та зычный притяжение взял физкультуры мушкетеров . тут был с привычке , кто все пять в ваню класс на мгновенно вечно пять партой вспыхнуло прикрепить же ещё пора лошадиных , твёрд начал , услышав родители , видишь , витьку всё займись два если вон бросил было решил костьми рухнул был за оба дедовым штурвал , но конец ведь в пионерку этот координат своей целым
